# 第8章　实战②　彩色图像分类（CIFAR-10）＋ 数据增强

第6章 MNIST（黑白・简单）的进阶。**CIFAR-10** 是 32×32 的**彩色照片**10类
（飞机・汽车・鸟・猫・鹿・狗・青蛙・马・船・卡车）。比 MNIST 难得多，
正好用来学**彩色（3通道）・数据增强・更深的 CNN・防过拟合**。

目标：搭彩色图像 CNN，用数据增强和 Dropout/BatchNorm 体会提升精度的感觉。

> **使用方法**：从上到下 `Shift + Enter`。推荐 GPU 的章节（Colab：代码执行程序→更改运行时类型→GPU）。

In [ ]:
import torch
print("PyTorch:", torch.__version__, "| GPU:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 8-1. 与 MNIST 的区别

| | MNIST | CIFAR-10 |
|---|---|---|
| 图像 | 黑白 28×28 | **彩色 32×32** |
| 形状 | `(N, 1, 28, 28)` | **`(N, 3, 32, 32)`**（C=3=RGB） |
| 难度 | 简单（可达99%） | 难（简单 CNN 75〜82%） |

要点：输入通道从 **1→3**，所以第一个 `Conv2d` 的 `in_channels=3`。

## 8-2. 数据增强（Data Augmentation）

把训练图像**随机轻微变形**（裁剪・左右翻转等）来扩充，能让模型学到"位置或朝向稍有不同也是同一只猫"，
从而**更不易过拟合**。

- **只对训练集**做增强。**测试集保持原样**（评估要公平）。
- `Normalize` 的均值・标准差用 CIFAR-10 的常用值。

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

mean = (0.4914, 0.4822, 0.4465)
std  = (0.2470, 0.2435, 0.2616)

train_tf = transforms.Compose([
    transforms.RandomCrop(32, padding=4),     # 补4px边后随机裁剪
    transforms.RandomHorizontalFlip(),        # 左右翻转
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
test_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_ds = datasets.CIFAR10("./data", train=True,  download=True, transform=train_tf)
test_ds  = datasets.CIFAR10("./data", train=False, download=True, transform=test_tf)
train_loader = DataLoader(train_ds, batch_size=128, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=256, shuffle=False, num_workers=2)
print("train:", len(train_ds), " test:", len(test_ds))

### 看看图像（把归一化还原后显示）

In [ ]:
import matplotlib.pyplot as plt
classes = ['airplane','automobile','bird','cat','deer','dog','frog','horse','ship','truck']
m = torch.tensor(mean).view(3,1,1); s = torch.tensor(std).view(3,1,1)

xb, yb = next(iter(train_loader))
fig, axes = plt.subplots(1, 6, figsize=(11, 2))
for ax, i in zip(axes, range(6)):
    img = (xb[i]*s + m).clamp(0,1).permute(1,2,0)   # 还原归一化并转成 (H,W,C)
    ax.imshow(img); ax.set_title(classes[yb[i]], fontsize=9); ax.axis("off")
plt.show()

## 8-3. 更深的 CNN（含 BatchNorm + Dropout）

- **`BatchNorm2d`**：整理每层输出 → 训练更快更稳。
- **`Dropout`**：训练时随机丢弃神经元 → 防过拟合。
- Conv→BN→ReLU→Pool 叠三段，把 32×32 缩到 4×4。

In [ ]:
import torch.nn as nn

class CIFARNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1),  nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),  # 32->16
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),  # 16->8
            nn.Conv2d(64, 128, 3, padding=1),nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),  # 8->4
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(128*4*4, 256), nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 10),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

model = CIFARNet().to(device)
# 形状检查（假输入 2张）
dummy = torch.randn(2, 3, 32, 32).to(device)
print("输出 shape:", model(dummy).shape)   # (2, 10)

## 8-4. 训练（推荐 GPU）
和第6章一样的五步。`EPOCHS` 在 GPU 上 10〜20 可超 80%。CPU 上减到 2〜3 即可（精度会降）。

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimizer.step()
        running += loss.item()
    print(f"epoch {epoch+1:2d}: 平均loss = {running/len(train_loader):.4f}")

## 8-5. 测试精度

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        pred = model(xb).argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += yb.size(0)
print(f"测试准确率: {100*correct/total:.2f}%")

## 8-6. 过拟合（overfitting）

CIFAR-10 上容易出现"**训练精度高但测试上不去**"，这就是过拟合。
对策就是这次加入的 **数据增强・Dropout・BatchNorm**。即便如此，简单 CNN 的上限约在 80% 出头，
要再高需要 **ResNet** 等技巧（残差连接）和学习率调度（→ 练习）。

## 练习 8
1. **去掉**数据增强（把 `train_tf` 改成和 `test_tf` 一样）再训练，观察 train 与测试精度差（过拟合）变大。
2. 增大 `EPOCHS`，加上学习率调度 `torch.optim.lr_scheduler.CosineAnnealingLR`，冲更高精度。
3. 比较 `Dropout` 比例（0.3→0.5）或有无 `BatchNorm` 的差别。
4. 有余力就换成 `torchvision.models.resnet18(num_classes=10)` 比较（针对 CIFAR 调整第一层更好）。

In [ ]:
# 在这里写你自己的代码并运行
